# 00 — Setup & Verification

Welcome to **Building a Gemini-Level Model from Scratch**.

This notebook verifies that your environment is correctly configured before we begin building transformer components. Run every cell top-to-bottom — if everything prints ✓ at the end, you are ready.

## What this notebook covers
1. Python & package version checks
2. PyTorch device detection (CPU / CUDA / MPS)
3. Quick smoke-test of every `src/` module
4. End-to-end forward pass through the full model
5. Visualisation sanity check

## 1 — Python & Core Packages

In [ ]:
import sys
import importlib

print(f'Python {sys.version}\n')

required = {
    'torch': '2.0.0',
    'numpy': '1.24.0',
    'matplotlib': '3.7.0',
    'yaml': None,          # pyyaml
    'tqdm': '4.65.0',
}

all_ok = True
for pkg, min_ver in required.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        status = '✓'
    except ImportError:
        ver = 'MISSING'
        status = '✗'
        all_ok = False
    suffix = f'  (need >= {min_ver})' if min_ver else ''
    print(f'  {status} {pkg:<20} {ver}{suffix}')

print()
if all_ok:
    print('All required packages found.')
else:
    print('Some packages are missing. Run:  pip install -r requirements.txt')

## 2 — PyTorch Device

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'CUDA version    : {torch.version.cuda}')

mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
print(f'MPS available   : {mps_available}')

if torch.cuda.is_available():
    device = torch.device('cuda')
elif mps_available:
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'\nUsing device    : {device}')

# Quick tensor op to confirm device works
x = torch.randn(4, 4, device=device)
y = x @ x.T
print(f'Tensor op test  : ✓  ({y.shape} on {y.device})')

## 3 — Smoke Test: Source Modules

In [ ]:
import sys, os
# Make sure the project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

modules_to_test = [
    ('src.models.attention',       'MultiHeadAttention'),
    ('src.models.embeddings',      'TransformerEmbedding'),
    ('src.models.feedforward',     'FeedForward'),
    ('src.models.transformer_block', 'DecoderBlock'),
    ('src.models.transformer',     'TransformerLM'),
    ('src.training.trainer',       'Trainer'),
    ('src.generation.sampling',    'greedy_decode'),
    ('src.utils.visualization',    'plot_attention_weights'),
]

print('Module imports:')
all_ok = True
for module_path, symbol in modules_to_test:
    try:
        mod = importlib.import_module(module_path)
        assert hasattr(mod, symbol), f'{symbol} not found in {module_path}'
        print(f'  ✓ {module_path}.{symbol}')
    except Exception as e:
        print(f'  ✗ {module_path}  →  {e}')
        all_ok = False

print()
print('All modules OK ✓' if all_ok else 'Some imports failed — check the errors above.')

## 4 — End-to-End Forward Pass

We instantiate a **tiny** model (small enough for CPU) and run a forward pass to confirm the full stack works together.

In [ ]:
from src.models.transformer import TransformerLM, TransformerConfig

cfg = TransformerConfig(
    vocab_size=256,
    d_model=64,
    n_heads=4,
    n_layers=2,
    d_ff=128,
    max_seq_len=64,
    dropout=0.0,
    pos_encoding='sinusoidal',
    ffn_type='standard',
    norm_type='layernorm',
)

model = TransformerLM(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model created: {n_params:,} parameters')

# Dummy input: batch=2, seq_len=16
tokens = torch.randint(0, cfg.vocab_size, (2, 16), device=device)

model.eval()
with torch.no_grad():
    logits = model(tokens)

print(f'Input  shape  : {tokens.shape}')
print(f'Output shape  : {logits.shape}')   # expect (2, 16, vocab_size)
assert logits.shape == (2, 16, cfg.vocab_size), 'Unexpected output shape!'
print('\nForward pass ✓')

## 5 — Attention Weight Visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.models.attention import MultiHeadAttention

# Small standalone attention layer
attn = MultiHeadAttention(d_model=64, n_heads=4).to(device)
x = torch.randn(1, 8, 64, device=device)   # batch=1, seq=8
_, weights = attn(x, x, x, return_attention=True)
# weights: (1, n_heads, 8, 8)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for i, ax in enumerate(axes):
    im = ax.imshow(weights[0, i].detach().cpu().numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {i+1}')
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
    plt.colorbar(im, ax=ax)

plt.suptitle('Multi-Head Attention Weights (random input)', fontsize=13)
plt.tight_layout()
plt.show()
print('Visualisation ✓')

## 6 — GPU Memory Report (optional)

In [ ]:
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1e6
    reserved = torch.cuda.memory_reserved() / 1e6
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Allocated : {alloc:.1f} MB')
    print(f'Reserved  : {reserved:.1f} MB')
    print(f'Total GPU : {total:.1f} GB')
else:
    print('No GPU detected — using CPU throughout the workshop.')
    print('The small model exercises will still run comfortably on CPU.')

## All Done!

If every cell above printed ✓ without errors, your environment is ready.

**Next:** Head to `part1_evolution/01_rnn_limitations_starter.ipynb` to understand *why* we moved from RNNs to Transformers.